In [ ]:
!pip install youtube-transcript-api
!pip install pytube3
!pip install pafy
!pip install youtube_dl
!pip install yt-dlp

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
import yt_dlp

In [ ]:
def get_video_urls(playlist_url):
    ydl_opts = {
        'quiet': True,
        'extract_flat': True,  # Only extract URLs
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        playlist_info = ydl.extract_info(playlist_url, download=False)
        video_urls = [entry['url'] for entry in playlist_info['entries']]
        video_titles = [entry['title'] for entry in playlist_info['entries']]
        return video_urls, video_titles

In [ ]:

def getTranscript(playlist_url):
  video_urls, video_titles = get_video_urls(playlist_url)

  SPEECH_DICT = {}
  for i in range(0,len(video_urls)):
    url = video_urls[i]
    title = video_titles[i]
    print(title)

    # get video id
    video_id = url.split('https://www.youtube.com/watch?v=')[1]

    SPEECH_DICT[video_id] = {}
    SPEECH_DICT[video_id]['url'] = url
    SPEECH_DICT[video_id]['title'] = title
    try:
      # get all the list of transcripts available in diff languages
      transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

      # find the one which is in hindi
      transcript = transcript_list.find_transcript(['hi'])

      # translate the transcript to english
      translated_transcript = transcript.translate('en')
      eng_transcript = translated_transcript.fetch()

      # format the transcript to sentence string
      formatter = TextFormatter()
      formatted_transcript = formatter.format_transcript(eng_transcript).replace('\n',' ').replace('[appreciation]','').replace('[music]','')

      # append in the list of speeches
      SPEECH_DICT[video_id]['text'] = formatted_transcript
    except:
      try:
        # get all the list of transcripts available in diff languages
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

        # find the one which is in hindi
        transcript = transcript_list.find_transcript(['en'])

        # translate the transcript to english
        translated_transcript = transcript.translate('en')
        eng_transcript = translated_transcript.fetch()

        # format the transcript to sentence string
        formatter = TextFormatter()
        formatted_transcript = formatter.format_transcript(eng_transcript).replace('\n',' ').replace('[appreciation]','').replace('[music]','')

      # append in the list of speeches
        SPEECH_DICT[video_id]['text'] = formatted_transcript
      except:
        SPEECH_DICT[video_id]['text'] = 'No Transcript Available'

  return SPEECH_DICT

In [ ]:
playlist_urls = ["https://www.youtube.com/playlist?list=PLqtVCj5iilH4Ia-QrpzLPmU-I2kkKbMql",
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH7VO-yhOJk08TDHVenjvEQ3",#independence day speeches
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH7FPEj8iWVe2ZW-ADtOkwjK",#atal bihari vajpayee
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH5vp001YjQNIn7AgmHqwAnx",#apj abdul kalam
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH6EtIep58nXOf84EFXGw7fi",#ram nath kovid
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH5NtsLtRYjjASsY92n9qdp0",#pratibha patil
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH7qQM0desSK3HmwEcUDhuO0",#radha krishana
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH58HBRYbZYlXdRFU4uejG4m",#neelam sanjeeva reddy
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4QghSrtPnTrdgoGZyhcM22",#president speeches
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH6vh2rG-PcuQe_F2OiEyw8F",#morarji desai
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4r0InJt9XYTrEWRGU0ccG6",#lal bahadur shastri
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4yNuJ0c7paHwzh6HZG5ooV",#indira gandhi
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4SkGxWM7k8X6N1F3jZyh-Q",#jawaharlal nehru
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH60r9bvyWNbq5QFcF1z9Xs8",#election speeches
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4vmSDciJZhSRl2oTEJvGMi",#rajiv gandhi
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4_cmJ9CzdfWD5tCRMWMbml",#pranab mukherjee
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH6Zfa_OPKixsq7e__wxJTLj",#kr narayan
                 ]

In [ ]:
import six
import pandas as pd

dfs = []
for playlist_url in playlist_urls:
  speech_dict = getTranscript(playlist_url)
  urls = []
  titles = []
  text = []
  for key,val in six.iteritems(speech_dict):
    urls.append(val['url'])
    titles.append(val['title'])
    text.append(val['text'])
  df = pd.DataFrame({'url':urls,'title':titles,'text':text})
  dfs.append(df)


In [ ]:
dataset = pd.concat(dfs)
dataset.drop_duplicates(inplace = True)
dataset = dataset[~dataset.text.str.contains('No Transcript Available')]

In [ ]:
len(dataset)

0

In [ ]:
dataset['text'] = dataset['text'].str.replace('praise','').str.replace('Praise','').str.replace('Music','').str.replace('music','').str.replace('applause','').str.replace('Applause','').str.replace('laughter','').str.replace('Laughter','').str.replace('[','').str.replace(']','')
dataset["words"] = dataset["text"].apply(lambda n: len(n.split()))


In [ ]:
modi_speech = pd.read_csv('PM_Modi_speeches.csv')
modi_speech_en = modi_speech[modi_speech.lang == 'en'][['url','title','text','words']]
dataset = pd.concat([dataset, modi_speech_en])

In [ ]:
len(dataset)

491

In [ ]:
dataset.head()

,url,title,text,words
0,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address in the 15th Episode of ‘Mann Ki B...,"My dear countrymen, Namaskar.\nGenerally, this...",21619
1,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at inauguration of the College an...,Our country’s Agriculture Minister Shri Narend...,10128
2,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at seminar on Atmanirbhar Bharat ...,"My cabinet colleague, Shri Rajnath ji, Chief o...",8497
3,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address to the Nation from the ramparts o...,"My dear countrymen,\nCongratulations and many ...",50260
4,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at the Launch of ‘Transparent Tax...,The process of Structural Reforms going on in ...,11908


In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Required for newer versions of NLTK
from nltk.util import bigrams, trigrams
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import nltk
from nltk.util import bigrams, trigrams
from nltk.tokenize import word_tokenize

# Function to generate bigrams
def generate_bigrams(text):
    tokens = word_tokenize(text)
    return list(bigrams(tokens))

# Function to generate trigrams
def generate_trigrams(text):
    tokens = word_tokenize(text)
    return list(trigrams(tokens))

# Apply functions to the DataFrame
dataset['bigrams'] = dataset['text'].apply(generate_bigrams)
dataset['trigrams'] = dataset['text'].apply(generate_trigrams)

dataset.head()


,url,title,text,words,bigrams,trigrams
0,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address in the 15th Episode of ‘Mann Ki B...,"My dear countrymen, Namaskar.\nGenerally, this...",21619,"[(My, dear), (dear, countrymen), (countrymen, ...","[(My, dear, countrymen), (dear, countrymen, ,)..."
1,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at inauguration of the College an...,Our country’s Agriculture Minister Shri Narend...,10128,"[(Our, country), (country, ’), (’, s), (s, Agr...","[(Our, country, ’), (country, ’, s), (’, s, Ag..."
2,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at seminar on Atmanirbhar Bharat ...,"My cabinet colleague, Shri Rajnath ji, Chief o...",8497,"[(My, cabinet), (cabinet, colleague), (colleag...","[(My, cabinet, colleague), (cabinet, colleague..."
3,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address to the Nation from the ramparts o...,"My dear countrymen,\nCongratulations and many ...",50260,"[(My, dear), (dear, countrymen), (countrymen, ...","[(My, dear, countrymen), (dear, countrymen, ,)..."
4,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at the Launch of ‘Transparent Tax...,The process of Structural Reforms going on in ...,11908,"[(The, process), (process, of), (of, Structura...","[(The, process, of), (process, of, Structural)..."


In [ ]:
dataset.to_csv('political_speech_dataset.csv')